In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

In [2]:
def find_repo_root(start: Path) -> Path:
    """Walk upward from the current directory until the repo root is found."""
    for candidate in [start, *start.parents]:
        if (candidate / "Results_Daily").exists() and (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current notebook directory.")


SIGNIFICANCE_LEVEL = 0.10

ASSET_CONFIG = {
    "QQQ": {
        "display_name": "NASDAQ 100",
        "runs": {
            "LSTM-1": "QQQ_LSTM_DAILY_1",
            "LSTM-2": "QQQ_LSTM_DAILY_2",
            "Transformer": "QQQ_TRANSFORMERS_DAILY_1",
            "LSTM-NC-1": "QQQ_LSTM_NC_DAILY",
            "LSTM-NC-2": "QQQ_LSTM_NC_DAILY_2",
        },
    },
    "NKY": {
        "display_name": "NIKKEI 225",
        "runs": {
            "LSTM-1": "NKY_LSTM_DAILY_1",
            "LSTM-2": "NKY_LSTM_DAILY_2",
            "Transformer": "NKY_TRANSFORMERS_DAILY_1",
            "LSTM-NC-1": "NKY_LSTM_NC_DAILY",
            "LSTM-NC-2": "NKY_LSTM_NC_DAILY_2",
        },
    },
    "EUSTX": {
        "display_name": "EURO STOXX 50",
        "runs": {
            "LSTM-1": "EUSTX_LSTM_DAILY_1",
            "LSTM-2": "EUSTX_LSTM_DAILY_2",
            "Transformer": "EUSTX_TRANSFORMERS_DAILY_1",
            "LSTM-NC-1": "EUSTX_LSTM_NC_DAILY",
            "LSTM-NC-2": "EUSTX_LSTM_NC_DAILY_2",
        },
    },
}

# Buy-and-Hold is intentionally excluded per request.
BENCHMARK_COLUMNS = [
    "QQQ Buy-and-Hold"
    # "Equal-Weight Monthly",
    # "Inverse Volatility",
    # "Momentum Top-20%",
    # "Supervised + MVO",
]

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = find_repo_root(NOTEBOOK_DIR)
RESULTS_ROOT = REPO_ROOT / "Results_Daily"
OUTPUT_DIR = RESULTS_ROOT / "Statistical_Significance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Repository root: /Users/kamilkashif/Documents/University/Masters Thesis/MS-Thesis-Deep-RL-KK
Results root: /Users/kamilkashif/Documents/University/Masters Thesis/MS-Thesis-Deep-RL-KK/Results_Daily
Output directory: /Users/kamilkashif/Documents/University/Masters Thesis/MS-Thesis-Deep-RL-KK/Results_Daily/Statistical_Significance


In [3]:
def load_experiment_frame(asset_code: str, experiment_folder: str) -> pd.DataFrame:
    rl_path = RESULTS_ROOT / asset_code / experiment_folder / "rl_daily_returns_oos.csv"
    benchmark_path = RESULTS_ROOT / asset_code / experiment_folder / "daily_returns_oos.csv"

    rl_df = pd.read_csv(rl_path)
    benchmark_df = pd.read_csv(benchmark_path)

    # Normalize RL return columns across minor file-format differences.
    if "date" not in rl_df.columns:
        rl_df = rl_df.rename(columns={rl_df.columns[0]: "date"})
    if "RL Agent" not in rl_df.columns:
        candidate_cols = [c for c in rl_df.columns if c != "date"]
        if not candidate_cols:
            raise ValueError(f"No return column found in {rl_path}")
        rl_df = rl_df.rename(columns={candidate_cols[0]: "RL Agent"})

    rl_df["date"] = pd.to_datetime(rl_df["date"])
    benchmark_df["date"] = pd.to_datetime(benchmark_df["date"])

    use_cols = ["date", *BENCHMARK_COLUMNS]
    merged = (
        rl_df[["date", "RL Agent"]]
        .rename(columns={"RL Agent": "strategy_return"})
        .merge(benchmark_df[use_cols], on="date", how="inner")
        .dropna(subset=["strategy_return", *BENCHMARK_COLUMNS])
        .sort_values("date")
        .reset_index(drop=True)
    )

    return merged


experiment_data_by_asset = {}
coverage_rows = []

for asset_code, asset_cfg in ASSET_CONFIG.items():
    experiment_data_by_asset[asset_code] = {}
    for model_label, folder_name in asset_cfg["runs"].items():
        df_model = load_experiment_frame(asset_code, folder_name)
        experiment_data_by_asset[asset_code][model_label] = df_model

        coverage_rows.append(
            {
                "Asset": asset_cfg["display_name"],
                "Model": model_label,
                "N": len(df_model),
                "Start": df_model["date"].min().date(),
                "End": df_model["date"].max().date(),
            }
        )

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df.sort_values(["Asset", "Model"]).reset_index(drop=True))

,Asset,Model,N,Start,End
0,EURO STOXX 50,LSTM-1,4085,2009-04-01,2025-03-28
1,EURO STOXX 50,LSTM-2,4085,2009-04-01,2025-03-28
2,EURO STOXX 50,LSTM-NC-1,4085,2009-04-01,2025-03-28
3,EURO STOXX 50,LSTM-NC-2,4085,2009-04-01,2025-03-28
4,EURO STOXX 50,Transformer,4085,2009-04-01,2025-03-28
5,NASDAQ 100,LSTM-1,4008,2009-04-06,2025-04-01
6,NASDAQ 100,LSTM-2,4008,2009-04-06,2025-04-01
7,NASDAQ 100,LSTM-NC-1,4008,2009-04-06,2025-04-01
8,NASDAQ 100,LSTM-NC-2,4008,2009-04-06,2025-04-01
9,NASDAQ 100,Transformer,4008,2009-04-06,2025-04-01


In [11]:
def bold_if_significant(value: float) -> str:
    if pd.notna(value) and value < SIGNIFICANCE_LEVEL:
        return "font-weight: bold"
    return ""

In [12]:
# --- Robust inference extension: Newey-West (HAC) + Stationary Bootstrap ---
# This section addresses serial correlation / heteroskedasticity concerns raised by the reviewer.

import math
import statsmodels.api as sm

ANNUALIZATION = 252
N_BOOTSTRAP = 1000  # increase to 2000+ for final tables if runtime allows
SB_AVG_BLOCK_LEN = 20
BOOTSTRAP_SEED = 42


def choose_newey_west_lags(n_obs: int) -> int:
    """Andrews-style plug-in lag rule (common practical default)."""
    if n_obs <= 1:
        return 1
    return max(1, int(math.floor(4.0 * (n_obs / 100.0) ** (2.0 / 9.0))))


def sign_nonzero(x: float) -> int:
    if x == 0:
        return 0
    return int(abs(x) / x)


def sharpe_annualized(returns: np.ndarray, annualization: int = 252) -> float:
    r = np.asarray(returns, dtype=float)
    if r.size < 2:
        return 0.0
    std = np.std(r)
    if std <= 1e-12:
        return 0.0
    return float(np.mean(r) / std * np.sqrt(annualization))


def ir2_from_returns(returns: np.ndarray, annualization: int = 252) -> float:
    """IR2 implementation aligned with Code/functions/baseline.py logic."""
    r = np.asarray(returns, dtype=float)
    if r.size == 0:
        return 0.0

    tab = np.concatenate(([1.0], np.cumprod(1.0 + r)))
    ret = (tab[1:] / tab[:-1]) - 1.0

    if len(tab) <= 1:
        return 0.0

    # ARC (%): mirrors baseline.py using ret[:-1]
    arc_base = np.prod(1.0 + ret[:-1]) if ret.size > 1 else 1.0
    if arc_base <= 0:
        arc = 0.0
    else:
        arc = 100.0 * (arc_base ** (annualization / len(tab)) - 1.0)

    # ASD (%)
    asd = float(np.sqrt(annualization) * np.std(ret) * 100.0) if ret.size else 0.0

    # MDD (%)
    if ret.size:
        cum_returns = np.cumprod(1.0 + ret)
        cum_max = np.maximum.accumulate(cum_returns)
        drawdowns = (cum_max - cum_returns) / cum_max
        mdd = float(np.max(drawdowns) * 100.0)
    else:
        mdd = 0.0

    denom = asd * mdd
    if denom == 0:
        return 0.0

    numer = (arc ** 2) * sign_nonzero(arc)
    return float(max(numer / denom, 0.0))


def stationary_bootstrap_indices(n_obs: int, restart_prob: float, rng: np.random.Generator) -> np.ndarray:
    idx = np.empty(n_obs, dtype=int)
    idx[0] = rng.integers(0, n_obs)
    for t in range(1, n_obs):
        if rng.random() < restart_prob:
            idx[t] = rng.integers(0, n_obs)
        else:
            idx[t] = (idx[t - 1] + 1) % n_obs
    return idx


def stationary_bootstrap_right_tail_pvalue(
    strategy_returns: np.ndarray,
    benchmark_returns: np.ndarray,
    stat_fn,
    n_bootstrap: int = 1000,
    avg_block_len: int = 20,
    seed: int = 42,
):
    """
    One-sided bootstrap p-value for H1: stat(strategy, benchmark) > 0.
    p-value computed as tail mass at/below zero in bootstrap distribution.
    """
    s = np.asarray(strategy_returns, dtype=float)
    b = np.asarray(benchmark_returns, dtype=float)
    n_obs = len(s)

    if n_obs != len(b):
        raise ValueError("Strategy and benchmark lengths must match.")
    if n_obs < 3:
        return np.nan, np.nan, (np.nan, np.nan, np.nan)

    obs_stat = float(stat_fn(s, b))

    rng = np.random.default_rng(seed)
    restart_prob = 1.0 / max(float(avg_block_len), 1.0)
    bootstrap_stats = np.empty(n_bootstrap, dtype=float)

    for i in range(n_bootstrap):
        idx = stationary_bootstrap_indices(n_obs, restart_prob, rng)
        bootstrap_stats[i] = stat_fn(s[idx], b[idx])

    p_right = (1.0 + np.sum(bootstrap_stats <= 0.0)) / (n_bootstrap + 1.0)
    q05, q50, q95 = np.quantile(bootstrap_stats, [0.05, 0.50, 0.95])
    return obs_stat, float(p_right), (float(q05), float(q50), float(q95))


def nw_mean_difference_test(diff_series: np.ndarray):
    """HAC test for H1: mean(diff_series) > 0."""
    d = np.asarray(diff_series, dtype=float)
    d = d[np.isfinite(d)]
    n_obs = len(d)
    if n_obs < 3:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    max_lags = choose_newey_west_lags(n_obs)
    X = np.ones((n_obs, 1), dtype=float)
    fit = sm.OLS(d, X).fit(cov_type="HAC", cov_kwds={"maxlags": max_lags})

    mean_diff = float(fit.params[0])
    se_hac = float(fit.bse[0])
    t_hac = float(fit.tvalues[0])
    p_right_hac = float(1.0 - stats.norm.cdf(t_hac))
    return mean_diff, se_hac, t_hac, p_right_hac, max_lags


def hac_alpha_regression_test(diff_series: np.ndarray, benchmark_returns: np.ndarray):
    """HAC OLS for diff_t = alpha + beta * benchmark_t + eps_t."""
    y = np.asarray(diff_series, dtype=float)
    x = np.asarray(benchmark_returns, dtype=float)

    mask = np.isfinite(y) & np.isfinite(x)
    y = y[mask]
    x = x[mask]
    n_obs = len(y)

    if n_obs < 3:
        return {
            "N": n_obs,
            "alpha": np.nan,
            "SE(alpha)_HAC": np.nan,
            "t(alpha)_HAC": np.nan,
            "p(alpha)_HAC_right": np.nan,
            "beta": np.nan,
            "SE(beta)_HAC": np.nan,
            "t(beta)_HAC": np.nan,
            "p(beta)_HAC_two_sided": np.nan,
            "NW_lags": np.nan,
        }

    max_lags = choose_newey_west_lags(n_obs)
    X = sm.add_constant(x)
    fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": max_lags})

    alpha = float(fit.params[0])
    beta = float(fit.params[1])
    t_alpha = float(fit.tvalues[0])
    t_beta = float(fit.tvalues[1])

    return {
        "N": n_obs,
        "alpha": alpha,
        "SE(alpha)_HAC": float(fit.bse[0]),
        "t(alpha)_HAC": t_alpha,
        "p(alpha)_HAC_right": float(1.0 - stats.norm.cdf(t_alpha)),
        "beta": beta,
        "SE(beta)_HAC": float(fit.bse[1]),
        "t(beta)_HAC": t_beta,
        "p(beta)_HAC_two_sided": float(2.0 * (1.0 - stats.norm.cdf(abs(t_beta)))),
        "NW_lags": max_lags,
    }


# --- Run robust tests across all assets/models/benchmarks ---
robust_rows = []
hac_regression_rows = []

for asset_code, asset_cfg in ASSET_CONFIG.items():
    for model_label, df_model in experiment_data_by_asset[asset_code].items():
        for benchmark in BENCHMARK_COLUMNS:
            aligned = df_model[["strategy_return", benchmark]].dropna().copy()
            strategy = aligned["strategy_return"].to_numpy(dtype=float)
            bench = aligned[benchmark].to_numpy(dtype=float)
            diff = strategy - bench
            n_obs = len(diff)

            if n_obs < 3:
                continue

            # Legacy paired t-test (for side-by-side comparison)
            naive_test = stats.ttest_rel(strategy, bench, alternative="greater")

            # HAC mean-difference test
            mean_diff, se_hac, t_hac, p_hac, nw_lags = nw_mean_difference_test(diff)

            # Stationary bootstrap p-values for Sharpe and IR2 differences
            sharpe_diff, p_sharpe_sb, sharpe_q = stationary_bootstrap_right_tail_pvalue(
                strategy,
                bench,
                stat_fn=lambda rs, rb: sharpe_annualized(rs, ANNUALIZATION) - sharpe_annualized(rb, ANNUALIZATION),
                n_bootstrap=N_BOOTSTRAP,
                avg_block_len=SB_AVG_BLOCK_LEN,
                seed=BOOTSTRAP_SEED,
            )

            ir2_diff, p_ir2_sb, ir2_q = stationary_bootstrap_right_tail_pvalue(
                strategy,
                bench,
                stat_fn=lambda rs, rb: ir2_from_returns(rs, ANNUALIZATION) - ir2_from_returns(rb, ANNUALIZATION),
                n_bootstrap=N_BOOTSTRAP,
                avg_block_len=SB_AVG_BLOCK_LEN,
                seed=BOOTSTRAP_SEED,
            )

            robust_rows.append(
                {
                    "Asset": asset_cfg["display_name"],
                    "Asset Code": asset_code,
                    "Model": model_label,
                    "Benchmark": benchmark,
                    "N": n_obs,
                    "NW_lags": nw_lags,
                    "Mean(strategy - benchmark)": mean_diff,
                    "SE_HAC": se_hac,
                    "t_HAC": t_hac,
                    "p_HAC_right": p_hac,
                    "p_ttest_right_legacy": float(naive_test.pvalue),
                    "Delta Sharpe": sharpe_diff,
                    "p_SB_right_DeltaSharpe": p_sharpe_sb,
                    "DeltaSharpe_q05": sharpe_q[0],
                    "DeltaSharpe_q50": sharpe_q[1],
                    "DeltaSharpe_q95": sharpe_q[2],
                    "Delta IR2": ir2_diff,
                    "p_SB_right_DeltaIR2": p_ir2_sb,
                    "DeltaIR2_q05": ir2_q[0],
                    "DeltaIR2_q50": ir2_q[1],
                    "DeltaIR2_q95": ir2_q[2],
                    "Significant_HAC_10pct": bool(pd.notna(p_hac) and p_hac < SIGNIFICANCE_LEVEL),
                    "Significant_SB_Sharpe_10pct": bool(pd.notna(p_sharpe_sb) and p_sharpe_sb < SIGNIFICANCE_LEVEL),
                    "Significant_SB_IR2_10pct": bool(pd.notna(p_ir2_sb) and p_ir2_sb < SIGNIFICANCE_LEVEL),
                }
            )

            # HAC alpha regression test
            hac_alpha = hac_alpha_regression_test(diff, bench)
            hac_regression_rows.append(
                {
                    "Asset": asset_cfg["display_name"],
                    "Asset Code": asset_code,
                    "Model": model_label,
                    "Benchmark": benchmark,
                    **hac_alpha,
                    "Significant_alpha_HAC_10pct": bool(
                        pd.notna(hac_alpha["p(alpha)_HAC_right"]) and hac_alpha["p(alpha)_HAC_right"] < SIGNIFICANCE_LEVEL
                    ),
                }
            )

robust_results = pd.DataFrame(robust_rows).sort_values(["Asset", "Model", "Benchmark"]).reset_index(drop=True)
hac_regression_results = pd.DataFrame(hac_regression_rows).sort_values(["Asset", "Model", "Benchmark"]).reset_index(drop=True)

print(
    f"Robust tests complete | bootstrap={N_BOOTSTRAP}, avg_block_len={SB_AVG_BLOCK_LEN}, alpha={SIGNIFICANCE_LEVEL:.2f}"
)
display(robust_results.head())
display(hac_regression_results.head())


# --- Ensemble robust tests (Results_Daily/Ensemble) ---
ensemble_panel_path = RESULTS_ROOT / "Ensemble" / "portfolio_returns_panel.csv"
ensemble_df = pd.read_csv(ensemble_panel_path)
if "date" in ensemble_df.columns:
    ensemble_df["date"] = pd.to_datetime(ensemble_df["date"])

ensemble_benchmark_col = "Benchmark"
ensemble_model_cols = [
    c for c in ensemble_df.columns
    if c not in {"date", ensemble_benchmark_col}
]

ensemble_robust_rows = []
ensemble_hac_regression_rows = []

for model_col in ensemble_model_cols:
    aligned = ensemble_df[[model_col, ensemble_benchmark_col]].dropna().copy()
    strategy = aligned[model_col].to_numpy(dtype=float)
    bench = aligned[ensemble_benchmark_col].to_numpy(dtype=float)
    diff = strategy - bench
    n_obs = len(diff)

    if n_obs < 3:
        continue

    naive_test = stats.ttest_rel(strategy, bench, alternative="greater")

    mean_diff, se_hac, t_hac, p_hac, nw_lags = nw_mean_difference_test(diff)

    sharpe_diff, p_sharpe_sb, sharpe_q = stationary_bootstrap_right_tail_pvalue(
        strategy,
        bench,
        stat_fn=lambda rs, rb: sharpe_annualized(rs, ANNUALIZATION) - sharpe_annualized(rb, ANNUALIZATION),
        n_bootstrap=N_BOOTSTRAP,
        avg_block_len=SB_AVG_BLOCK_LEN,
        seed=BOOTSTRAP_SEED,
    )

    ir2_diff, p_ir2_sb, ir2_q = stationary_bootstrap_right_tail_pvalue(
        strategy,
        bench,
        stat_fn=lambda rs, rb: ir2_from_returns(rs, ANNUALIZATION) - ir2_from_returns(rb, ANNUALIZATION),
        n_bootstrap=N_BOOTSTRAP,
        avg_block_len=SB_AVG_BLOCK_LEN,
        seed=BOOTSTRAP_SEED,
    )

    ensemble_robust_rows.append(
        {
            "Asset": "Ensemble",
            "Asset Code": "ENS",
            "Model": model_col,
            "Benchmark": ensemble_benchmark_col,
            "N": n_obs,
            "NW_lags": nw_lags,
            "Mean(strategy - benchmark)": mean_diff,
            "SE_HAC": se_hac,
            "t_HAC": t_hac,
            "p_HAC_right": p_hac,
            "p_ttest_right_legacy": float(naive_test.pvalue),
            "Delta Sharpe": sharpe_diff,
            "p_SB_right_DeltaSharpe": p_sharpe_sb,
            "DeltaSharpe_q05": sharpe_q[0],
            "DeltaSharpe_q50": sharpe_q[1],
            "DeltaSharpe_q95": sharpe_q[2],
            "Delta IR2": ir2_diff,
            "p_SB_right_DeltaIR2": p_ir2_sb,
            "DeltaIR2_q05": ir2_q[0],
            "DeltaIR2_q50": ir2_q[1],
            "DeltaIR2_q95": ir2_q[2],
            "Significant_HAC_10pct": bool(pd.notna(p_hac) and p_hac < SIGNIFICANCE_LEVEL),
            "Significant_SB_Sharpe_10pct": bool(pd.notna(p_sharpe_sb) and p_sharpe_sb < SIGNIFICANCE_LEVEL),
            "Significant_SB_IR2_10pct": bool(pd.notna(p_ir2_sb) and p_ir2_sb < SIGNIFICANCE_LEVEL),
        }
    )

    hac_alpha = hac_alpha_regression_test(diff, bench)
    ensemble_hac_regression_rows.append(
        {
            "Asset": "Ensemble",
            "Asset Code": "ENS",
            "Model": model_col,
            "Benchmark": ensemble_benchmark_col,
            **hac_alpha,
            "Significant_alpha_HAC_10pct": bool(
                pd.notna(hac_alpha["p(alpha)_HAC_right"]) and hac_alpha["p(alpha)_HAC_right"] < SIGNIFICANCE_LEVEL
            ),
        }
    )

ensemble_robust_results = pd.DataFrame(ensemble_robust_rows).sort_values(["Model", "Benchmark"]).reset_index(drop=True)
ensemble_hac_regression_results = pd.DataFrame(ensemble_hac_regression_rows).sort_values(["Model", "Benchmark"]).reset_index(drop=True)

print("Ensemble robust tests complete")
display(ensemble_robust_results)
display(ensemble_hac_regression_results)


# --- Save robust outputs ---
robust_results.to_csv(OUTPUT_DIR / "paired_hac_stationary_bootstrap_all_assets.csv", index=False)
hac_regression_results.to_csv(OUTPUT_DIR / "alpha_regression_hac_all_assets.csv", index=False)

for asset_code, asset_cfg in ASSET_CONFIG.items():
    asset_slug = asset_cfg["display_name"].lower().replace(" ", "_")

    robust_results.loc[robust_results["Asset Code"] == asset_code].to_csv(
        OUTPUT_DIR / f"paired_hac_stationary_bootstrap_{asset_slug}.csv", index=False
    )
    hac_regression_results.loc[hac_regression_results["Asset Code"] == asset_code].to_csv(
        OUTPUT_DIR / f"alpha_regression_hac_{asset_slug}.csv", index=False
    )

ensemble_robust_results.to_csv(
    OUTPUT_DIR / "paired_hac_stationary_bootstrap_ensemble.csv", index=False
)
ensemble_hac_regression_results.to_csv(
    OUTPUT_DIR / "alpha_regression_hac_ensemble.csv", index=False
)

print(f"Saved robust results (assets + ensemble) to: {OUTPUT_DIR}")

Robust tests complete | bootstrap=1000, avg_block_len=20, alpha=0.10


,Asset,Asset Code,Model,Benchmark,N,NW_lags,Mean(strategy - benchmark),SE_HAC,t_HAC,p_HAC_right,...,DeltaSharpe_q50,DeltaSharpe_q95,Delta IR2,p_SB_right_DeltaIR2,DeltaIR2_q05,DeltaIR2_q50,DeltaIR2_q95,Significant_HAC_10pct,Significant_SB_Sharpe_10pct,Significant_SB_IR2_10pct
0,EURO STOXX 50,EUSTX,LSTM-1,QQQ Buy-and-Hold,4085,9,2.236128e-07,0.000109,0.002047,0.499183,...,0.111510,0.291462,0.059732,0.134865,-0.026596,0.048869,0.235942,False,False,False
1,EURO STOXX 50,EUSTX,LSTM-2,QQQ Buy-and-Hold,4085,9,-2.471463e-05,0.000116,-0.212209,0.584028,...,0.163406,0.346307,0.088636,0.061938,-0.001418,0.077368,0.328238,False,True,True
2,EURO STOXX 50,EUSTX,LSTM-NC-1,QQQ Buy-and-Hold,4085,9,-3.938654e-05,0.000122,-0.322176,0.626340,...,0.031052,0.242901,0.020483,0.378621,-0.084826,0.010344,0.148886,False,False,False
3,EURO STOXX 50,EUSTX,LSTM-NC-2,QQQ Buy-and-Hold,4085,9,9.469452e-06,0.000113,0.083999,0.466529,...,0.103989,0.296658,0.054037,0.160839,-0.041064,0.041761,0.239102,False,False,False
4,EURO STOXX 50,EUSTX,Transformer,QQQ Buy-and-Hold,4085,9,6.649134e-06,0.000109,0.060780,0.475767,...,0.118337,0.300656,0.064848,0.122877,-0.020893,0.053037,0.244715,False,False,False


,Asset,Asset Code,Model,Benchmark,N,alpha,SE(alpha)_HAC,t(alpha)_HAC,p(alpha)_HAC_right,beta,SE(beta)_HAC,t(beta)_HAC,p(beta)_HAC_two_sided,NW_lags,Significant_alpha_HAC_10pct
0,EURO STOXX 50,EUSTX,LSTM-1,QQQ Buy-and-Hold,4085,0.000163,0.000089,1.834612,0.033282,-0.421679,0.013809,-30.536824,0.0,9,True
1,EURO STOXX 50,EUSTX,LSTM-2,QQQ Buy-and-Hold,4085,0.000171,0.000076,2.258712,0.011951,-0.505843,0.011697,-43.245524,0.0,9,True
2,EURO STOXX 50,EUSTX,LSTM-NC-1,QQQ Buy-and-Hold,4085,0.000121,0.000105,1.153530,0.124346,-0.414051,0.014797,-27.982195,0.0,9,False
3,EURO STOXX 50,EUSTX,LSTM-NC-2,QQQ Buy-and-Hold,4085,0.000166,0.000096,1.731573,0.041675,-0.404842,0.014086,-28.741194,0.0,9,True
4,EURO STOXX 50,EUSTX,Transformer,QQQ Buy-and-Hold,4085,0.000169,0.000089,1.893557,0.029142,-0.420036,0.013872,-30.278369,0.0,9,True


Ensemble robust tests complete


,Asset,Asset Code,Model,Benchmark,N,NW_lags,Mean(strategy - benchmark),SE_HAC,t_HAC,p_HAC_right,...,DeltaSharpe_q50,DeltaSharpe_q95,Delta IR2,p_SB_right_DeltaIR2,DeltaIR2_q05,DeltaIR2_q50,DeltaIR2_q95,Significant_HAC_10pct,Significant_SB_Sharpe_10pct,Significant_SB_IR2_10pct
0,Ensemble,ENS,LSTM_1 (EW),Benchmark,3735,8,0.000001,0.000070,0.016880,0.493266,...,0.090836,0.288764,0.070520,0.253746,-0.134303,0.072134,0.435166,False,False,False
1,Ensemble,ENS,LSTM_2 (EW),Benchmark,3735,8,-0.000070,0.000073,-0.961978,0.831970,...,0.096957,0.272698,0.055276,0.245754,-0.129773,0.071114,0.417191,False,False,False
2,Ensemble,ENS,LSTM_NC_DAILY (EW),Benchmark,3735,8,-0.000002,0.000083,-0.024623,0.509822,...,0.009103,0.212083,0.006862,0.492507,-0.280261,0.001204,0.272001,False,False,False
3,Ensemble,ENS,LSTM_NC_DAILY_2 (EW),Benchmark,3735,8,-0.000006,0.000075,-0.085602,0.534109,...,0.020122,0.206033,0.011890,0.460539,-0.226500,0.009253,0.292806,False,False,False
4,Ensemble,ENS,TRANSFORMERS (EW),Benchmark,3735,8,-0.000028,0.000070,-0.393662,0.653085,...,0.040133,0.218546,0.021735,0.402597,-0.192402,0.024792,0.292584,False,False,False


,Asset,Asset Code,Model,Benchmark,N,alpha,SE(alpha)_HAC,t(alpha)_HAC,p(alpha)_HAC_right,beta,SE(beta)_HAC,t(beta)_HAC,p(beta)_HAC_two_sided,NW_lags,Significant_alpha_HAC_10pct
0,Ensemble,ENS,LSTM_1 (EW),Benchmark,3735,0.000110,0.000067,1.648776,0.049597,-0.205675,0.012686,-16.212186,0.0,8,True
1,Ensemble,ENS,LSTM_2 (EW),Benchmark,3735,0.000093,0.000056,1.663970,0.048059,-0.308218,0.010545,-29.228206,0.0,8,True
2,Ensemble,ENS,LSTM_NC_DAILY (EW),Benchmark,3735,0.000082,0.000083,0.992332,0.160518,-0.159580,0.016228,-9.833480,0.0,8,False
3,Ensemble,ENS,LSTM_NC_DAILY_2 (EW),Benchmark,3735,0.000079,0.000074,1.057654,0.145107,-0.161291,0.014743,-10.940385,0.0,8,False
4,Ensemble,ENS,TRANSFORMERS (EW),Benchmark,3735,0.000081,0.000066,1.213513,0.112467,-0.204745,0.012634,-16.205301,0.0,8,False


Saved robust results (assets + ensemble) to: /Users/kamilkashif/Documents/University/Masters Thesis/MS-Thesis-Deep-RL-KK/Results_Daily/Statistical_Significance
